# Kaggle: AutoML baselines (AutoGluon / LightAutoML / H2O)

**Accelerator:** 2× GPU T4.

## Порядок запуска
| Шаг | Ячейка | Действие |
|-----|--------|----------|
| 1 | §1–2 | Пути, `pip install`, GPU |
| 2 | §3 | Распаковка кода (**после каждого обновления репо**) |
| 3 | §4 | `HF_TOKEN` |
| 4 | §5 | `prepare_data` + функции `run_baseline` |
| 5 | §5b | Проверка файлов |
| 6 | §6a | **Пропуск** — AutoGluon уже в `metrics.json` (12 прогонов) |
| 7 | §6b | LightAutoML — 12 прогонов |
| 8 | §6c | H2O full — 3 seed |
| 9 | §7 | zip (опционально) |

**Output:** после каждого OK — `METRICS_JSON` в консоли ячейки. Скопируй логи в `results/buf.json` для слияния в `metrics.json`.


## 1. Пути


In [ ]:
import os, subprocess, sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
PROJECT_ROOT = WORK / "kaggle_jailbreak_automl"
TASK_ROOT = PROJECT_ROOT / "tasks" / "jailbreak_detection"
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))


## 2. Зависимости и GPU (2× T4)


In [ ]:
import subprocess, sys

pkgs = [
    "pandas>=2.0", "numpy>=1.24,<3", "scikit-learn>=1.5,<2", "joblib>=1.3",
    "datasets==3.6.0", "python-dotenv>=1,<2",
    "h2o>=3.46,<4", "autogluon.tabular>=1.1,<2", "lightautoml>=0.4.1,<0.5",
]
_pip = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True,
    check=False,
)
if _pip.returncode != 0:
    print(_pip.stderr or "", flush=True)
    _pip.check_returncode()
print("pip OK: h2o, autogluon.tabular, lightautoml", flush=True)

import torch
NUM_GPUS = int(torch.cuda.device_count())
assert NUM_GPUS >= 2, f"Нужно 2× T4, сейчас cuda devices={NUM_GPUS}"

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["JAILBREAK_NUM_GPUS"] = str(NUM_GPUS)
os.environ["JAILBREAK_METRICS_ONLY"] = "1"
os.environ["JAILBREAK_METRICS_OUTPUT_FILES"] = "0"
os.environ["JAILBREAK_QUIET_LOGS"] = "1"
for k, v in {
    "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1",
    "VECLIB_MAXIMUM_THREADS": "1", "NUMEXPR_NUM_THREADS": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "TRANSFORMERS_VERBOSITY": "error", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "TQDM_DISABLE": "1",
}.items():
    os.environ.setdefault(k, v)

# Память / лимиты (см. run_automl_baselines.py)
os.environ.setdefault("H2O_MAX_MEM", "14G")
os.environ.setdefault("H2O_MAX_MODELS", "16")
os.environ.setdefault("H2O_MAX_SEC_PER_MODEL", "600")
os.environ.setdefault("H2O_XGBOOST_GPU", "1")
os.environ.setdefault("AG_NUM_GPUS", "1")
os.environ.setdefault("JAILBREAK_LAMA_GPU_IDS", "0")
os.environ.setdefault("JAILBREAK_LAMA_MEMORY_LIMIT", "12")
os.environ.setdefault("JAILBREAK_TFIDF_MAX_FEATURES_FULL", "4000")
os.environ.setdefault("JAILBREAK_TFIDF_SVD_FULL", "128")
os.environ.setdefault("MALLOC_ARENA_MAX", "2")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("env OK", flush=True)


## 3. Код проекта (base64)


In [ ]:
import base64
import json
from pathlib import Path

_BLOB = """
...
ZXhjKClcbiAgICAgICAgcmFpc2VcbiJ9
"""

_P = json.loads(base64.b64decode(_BLOB.encode("ascii")).decode("utf-8"))
for rel, txt in _P.items():
    p = PROJECT_ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(txt, encoding="utf-8")
print("Written", len(_P), "files under", PROJECT_ROOT)


## 4. Hugging Face token


In [ ]:
import os

HF_TOKEN = ""  # <-- hf_... или Kaggle Secret HF_TOKEN

def apply_hf_token():
    token = (HF_TOKEN or "").strip()
    if not token and IS_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    if not token:
        raise ValueError("HF_TOKEN")
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token

apply_hf_token()
print("HF token OK", flush=True)


## 5. Данные и run_baseline


In [ ]:
def run_script(rel, extra=None, *, quiet=True, stderr_quiet=False):
    cmd = [sys.executable, "-u", str(PROJECT_ROOT / rel)] + (extra or [])
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    pp = str(PROJECT_ROOT)
    env["PYTHONPATH"] = pp if not env.get("PYTHONPATH") else pp + os.pathsep + env["PYTHONPATH"]
    env.setdefault("MALLOC_ARENA_MAX", "2")
    kw = {}
    if quiet:
        kw["stdout"] = subprocess.DEVNULL
        kw["stderr"] = subprocess.DEVNULL
    elif stderr_quiet:
        kw["stderr"] = subprocess.DEVNULL
    r = subprocess.run(cmd, check=False, cwd=str(PROJECT_ROOT), env=env, **kw)
    if r.returncode != 0:
        subprocess.run(cmd, check=True, cwd=str(PROJECT_ROOT), env=env, **kw)

print("prepare_data (few-shot)...", flush=True)
run_script("tasks/jailbreak_detection/scripts/prepare_data.py", quiet=True)
print("prepare_data --full_subset...", flush=True)
run_script("tasks/jailbreak_detection/scripts/prepare_data.py", ["--full_subset"], quiet=True)

SEEDS = [42, 123, 456]

def _print_metrics_stdout(stdout: str) -> None:
    marker = "METRICS_JSON"
    if marker not in stdout:
        if stdout.strip():
            print(stdout.strip(), flush=True)
        return
    print(stdout[stdout.index(marker) :].strip(), flush=True)

def run_baseline(fw, mode, seed, n_shots=None, tlim=900):
    label = f"{fw} {mode} seed={seed}"
    if mode == "fewshot":
        label += f" n_shots={n_shots}"
    print(f"\n>>> RUN {label}", flush=True)
    extra = [
        "--framework", fw, "--mode", mode, "--seed", str(seed),
        "--time-limit-sec", str(tlim), "--metrics-only", "--no-save-files",
    ]
    if mode == "fewshot":
        extra += ["--n_shots", str(n_shots)]
    cmd = [sys.executable, "-u", str(PROJECT_ROOT / "tasks/jailbreak_detection/scripts/run_automl_baselines.py")] + extra
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    pp = str(PROJECT_ROOT)
    env["PYTHONPATH"] = pp if not env.get("PYTHONPATH") else pp + os.pathsep + env["PYTHONPATH"]
    env.setdefault("MALLOC_ARENA_MAX", "2")
    proc = subprocess.run(
        cmd,
        cwd=str(PROJECT_ROOT),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )
    if proc.returncode != 0:
        if proc.stdout:
            print(proc.stdout, flush=True)
        if proc.stderr:
            print("--- stderr ---", flush=True)
            print(proc.stderr, flush=True)
        proc.check_returncode()
    _print_metrics_stdout(proc.stdout or "")
    print(f">>> OK {label}\n", flush=True)
    import gc
    gc.collect()


## 5b. Проверка данных


In [ ]:
proc = TASK_ROOT / "data" / "processed"
need = ["test.json", "wildjailbreak_eval_binary.jsonl", "wildjailbreak_full100k_seed42.json", "train_shot10_seed42.json"]
for f in need:
    p = proc / f
    assert p.is_file(), f"missing {p}"
print("data OK:", proc, flush=True)


## 6a. AutoGluon — пропуск (уже в metrics.json)


In [ ]:
# 12 прогонов autogluon_automl_baseline уже в metrics.json (2026-05-21)
print("skip §6a autogluon (done)", flush=True)


## 6b. LightAutoML — few-shot + full


In [ ]:
LM_FEW, LM_FULL = 900, 3600
for n in [10, 20, 50]:
    for seed in SEEDS:
        run_baseline("lightautoml", "fewshot", seed, n, tlim=LM_FEW)
for seed in SEEDS:
    run_baseline("lightautoml", "full", seed, tlim=LM_FULL)


## 6c. H2O — full × 3 seeds

Лимит **2400 с**/seed. OOM → в §2: `H2O_MAX_MEM=12G`, `H2O_TIME=1800`.


In [ ]:
H2O_TIME = 2400
for seed in SEEDS:
    run_baseline("h2o", "full", seed, tlim=H2O_TIME)


## 7. Артефакты (опционально)


In [ ]:
import shutil
z = WORK / "kaggle_jailbreak_automl_bundle.zip"
if z.is_file():
    z.unlink()
shutil.make_archive(str(WORK / "kaggle_jailbreak_automl_bundle"), "zip", PROJECT_ROOT)
print("zip:", z, flush=True)
